In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from joblib import Parallel, delayed
from tqdm import tqdm
import time
from pixell import enmap
import copy
import matplotlib
from scipy import special, optimize, integrate, stats
# import classy module
from classy import Class # type: ignore
from scipy import integrate
# from cosmoprimo.fiducial import DESI # tienes que tener el environment de cosmodesi # type: ignore
import pandas as pd
import pyclass # type: ignore
import yaml
import argparse

# Import pruning functions from remove_close_pairs
import sys
sys.path.append('./catalogue/')
from remove_close_pairs import enforce_min_separation, physical_to_min_sep, mean_redshift_from_table # type: ignore
import astropy.units as u
from astropy.table import Table as AstropyTable

In [2]:
path = '/pscratch/sd/j/jia_qu/ACTxDESIY3'
# Load the DESI Y3 LRG catalog
desi_lrg_catalog = Table.read(f'{path}/dr9_extended_lrg_pzbins.fits')

In [3]:
desi_lrg_catalog

TARGETID,RA,DEC,EBV,PIXEL_NOBS_G,PIXEL_NOBS_R,PIXEL_NOBS_Z,MASKBITS,PHOTSYS,Z_PHOT_MEDIAN,lrg_mask,pz_bin,LOGM
int64,float64,float64,float32,int16,int16,int16,int16,bytes1,float32,uint8,int16,float32
39633377069893749,88.876619891713,59.17631479529191,0.13461672,3,3,2,0,N,1.1000099,0,-1,-99.0
39633377069893759,88.87836001853711,59.235341111852534,0.13807458,3,3,2,0,N,0.7398904,0,3,11.460811
39633377069893796,88.88267213906238,59.1636643540421,0.13454334,3,3,2,0,N,0.6308099,0,2,10.990633
39633377069893895,88.89377281061189,59.12913677642326,0.13511845,3,3,2,0,N,0.94167084,0,4,10.898431
39633377069894013,88.90899344853247,59.19957280420695,0.1354234,3,3,2,0,N,0.534148,0,1,11.130031
39633377069894093,88.92062527481495,59.242353103805115,0.13784872,3,3,2,0,N,0.8954832,0,4,-99.0
39633377069894095,88.92068057067536,59.252385335143394,0.13856341,3,3,2,0,N,0.48074102,0,1,11.452891
39633377069894189,88.93331252223093,59.20544627082721,0.13566688,3,4,2,0,N,0.54189235,0,1,11.22756
39633377069894255,88.94076663346684,59.17332511268551,0.13520546,3,3,2,0,N,0.39187372,0,-1,11.428777


In [9]:
path2 = '/global/cfs/cdirs/desi/users/boryanah/reconstruction_DESI/recon/catalog_BGS_BRIGHT-20.2_R12.50_nmesh512_recsym_MG_cigale_masked.fits'
# path2 = '/global/cfs/cdirs/desi/users/boryanah/reconstruction_DESI/recon/catalog_BGS_BRIGHT-20.2_R12.50_nmesh512_recsym_MG_masked.fits'
desi_bgs_catalog = Table.read(path2)
len(desi_bgs_catalog)
desi_bgs_catalog

RA,DEC,Z,vR,LOGMSTAR
float64,float64,float64,float64,float32
126.89783846028135,-2.904775578004506,0.375015811350343,-70.52785375719351,10.641116
126.95986209930932,-2.878837205377212,0.24960681683539795,242.3144065987977,10.996213
126.97710447115486,-2.914306239078659,0.25052323454844777,210.6517602339666,11.023154
127.06331307478588,-2.943765335251469,0.19254384208503642,-124.90475925482298,9.800759
127.15439510540986,-2.901494029508148,0.24269274714440042,420.9649186591793,10.705479
127.17171690169056,-2.9025315409915686,0.24042289550881937,407.9686514599492,11.23532
127.17414953781943,-2.8795448117070697,0.31384243828076475,-204.52266256922096,10.435395
127.48980787812864,-2.9096382333770467,0.2293044597217506,265.40843158120833,10.510115
127.61751735146677,-2.9391694097037258,0.34775042974343967,-36.30124398286809,10.681143


In [7]:
path3 = '/global/cfs/cdirs/desi/survey/catalogs/DA2/analysis/loa-v1/LSScats/v1.1/BAO/unblinded/desipipe/2pt/recon_sm15_IFFT_recsym/'
desi_bgs2 = Table.read(f'{path3}/BGS_BRIGHT-20.2_NGC_clustering.dat.fits')
desi_bgs3 = Table.read(f'{path3}/BGS_BRIGHT-20.2_SGC_clustering.dat.fits')
# len(desi_bgs2), len(desi_bgs3)
len(desi_bgs2) + len(desi_bgs3)

4846786

In [ ]:
desi_bgs2

In [11]:
path4 = '/pscratch/sd/r/rhliu/projects/Weak_lensing/desi/spec_Y3/BGS_catalogues/full_catalog_Y3_no_src_with_cluster_mask.csv'
desi_bgs4 = pd.read_csv(path4)
len(desi_bgs4)
desi_bgs4

,TARGETID,Z,NTILE,RA,DEC,PHOTSYS,FRAC_TLOBS_TILES,WEIGHT_ZFAIL,WEIGHT_IMLIN,WEIGHT_FKP,...,flux_g_dered,flux_r_dered,flux_z_dered,flux_w1_dered,flux_w2_dered,NX,GC,DISP_LOS,VEL_LOS,VEL_LOS_RENORM
0,39628046080153028,0.010072,4,176.439922,10.824661,b'S',1.000000,1.000000,1.0,0.039129,...,6488.871000,13811.191000,25865.078000,18180.418000,10394.931000,0.003508,NGC,1.813787,64.618209,95.762893
1,39628128435306827,0.010114,3,159.160158,14.171101,b'S',0.995536,1.000000,1.0,0.041874,...,7931.457500,15844.148000,27300.883000,16108.229000,9464.590000,0.003269,NGC,0.954763,34.022905,50.420165
2,39627995324875346,0.010754,2,349.934219,8.576339,b'S',0.995215,1.000000,1.0,0.024025,...,6428.031700,12994.522000,23035.395000,14234.733000,8099.805000,0.005803,SGC,-7.163017,-255.889419,-379.086861
3,39628025171544366,0.011733,3,350.975179,9.667632,b'S',1.000000,1.000000,1.0,0.020909,...,4995.260700,9756.311000,16422.445000,8108.087400,4868.678000,0.006690,SGC,-6.858385,-245.136530,-362.969705
4,39627941348378526,0.011802,3,348.303093,6.321710,b'S',1.000000,1.000000,1.0,0.020909,...,4821.823000,9441.540000,16132.400000,11240.587000,7565.876000,0.006690,SGC,-7.461535,-266.747244,-394.954145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2249084,39628016363507551,0.499992,3,178.913299,9.624339,b'S',1.000000,1.024159,1.0,0.595146,...,4.237398,16.869402,38.236263,73.016660,48.817184,0.000097,NGC,7.700354,346.526630,397.244788
2249085,39627615530649126,0.499996,3,335.407821,-7.222935,b'S',1.000000,1.000087,1.0,0.580150,...,6.983283,30.798513,62.594170,80.290764,46.288020,0.000103,SGC,4.701947,211.724167,242.712006
2249086,39628204486428122,0.499997,2,186.337526,17.410612,b'S',0.993386,1.000687,1.0,0.628022,...,3.956078,21.513058,58.188790,114.555350,62.603836,0.000085,NGC,-1.463100,-65.965037,-75.619601
2249087,39627785534181098,0.499997,3,48.157499,-0.057858,b'S',1.000000,1.000000,1.0,0.580150,...,3.355390,16.267172,44.707157,83.143260,57.281643,0.000103,SGC,-2.902229,-130.887820,-150.044379


In [7]:
path = '/global/cfs/projectdirs/desi/users/mlokken/oriented_stacks/'
fileName = 'lrgc_nocut_lrgc_nocut_noorient_100pct_0.7_0.9.csv'
df = pd.read_csv(path + fileName, skiprows=1)

In [ ]:
df.columns

Index(['RA', 'DEC', 'Z'], dtype='object')

In [2]:
import sys
print(sys.path)

['/global/u2/r/rhliu/projects/repos/ThumbStack_kSZ/desi/spec_Y3', '/opt/nersc/pymon', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python310.zip', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10/lib-dynload', '', '/global/homes/r/rhliu/.local/lib/python3.10/site-packages', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10/site-packages', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10/site-packages/classy-3.2.1-py3.10-linux-x86_64.egg', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10/site-packages/pypolychord-1.22.0-py3.10-linux-x86_64.egg']


In [2]:
import sys, site
print(sys.executable)
print(sys.version)
print("USER_SITE:", site.getusersitepackages())
print("sys.path[0:5]:", sys.path[:5])

/global/homes/r/rhliu/myenvs/cosmodesi_dr1/bin/python
3.10.13 | packaged by conda-forge | (main, Oct 26 2023, 18:07:37) [GCC 12.3.0]
USER_SITE: /global/homes/r/rhliu/.local/lib/python3.10/site-packages
sys.path[0:5]: ['/global/u2/r/rhliu/projects/repos/ThumbStack_kSZ/desi/spec_Y3', '/opt/nersc/pymon', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python310.zip', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10', '/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20240118-1.0.0/conda/lib/python3.10/lib-dynload']
